# Clase 152 — RAG básico y embeddings

Un sistema **RAG** (Retrieval-Augmented Generation) enriquece a un LLM con
conocimiento externo: `embed(docs) → índice vectorial → retrieve top-k → prompt
con contexto → LLM`. Acá construimos el pipeline: embeddings, **similitud
coseno** (numpy, ejecutable-correcto), retrieval e inyección de contexto.

**Requiere:** ideal `sentence-transformers` + `faiss`. Si no están, usamos un
embedding **bag-of-words** en numpy (real y ejecutable) para que el retrieval
corra igual; los bloques con librerías pesadas van guardados.

## 🧠 Intuición previa

**RAG = buscar antes de responder.** En vez de confiar en lo que el modelo "recuerda" (y a veces inventa), primero se **buscan los pasajes más relevantes** de una base de documentos y se le pide al LLM que responda **citándolos**. Es como un examen a libro abierto: el modelo no adivina de memoria, sino que lee los fragmentos recuperados y compone la respuesta a partir de ellos (y admite *"no lo sé"* si el contexto no lo cubre). Esto reduce alucinaciones y permite actualizar el conocimiento **sin reentrenar** — basta con cambiar los documentos.

## 1. Entorno + corpus de documentos

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except Exception as e:
    HAS_ST = False
    print('sentence-transformers no instalado; uso embeddings bag-of-words. Motivo:', type(e).__name__)

import numpy as np
np.random.seed(42)

corpus = [
    'FAISS es una librería para búsqueda de vectores densos.',
    'BM25 es un método de recuperación disperso basado en frecuencias.',
    'Los embeddings mapean texto a vectores de dimensión fija.',
    'El re-ranking con cross-encoder mejora la precisión del top-k.',
    'Chroma y Qdrant son bases de datos vectoriales.',
    'Un LLM genera la respuesta a partir del contexto recuperado.',
]
print(f'{len(corpus)} documentos en el corpus')

## 2. Embeddings del corpus

Con `sentence-transformers` sería `SentenceTransformer('all-MiniLM-L6-v2').encode(...)`.
El fallback construye un vector de conteo de términos por documento — misma
mecánica (texto → vector) sobre la que opera la similitud.

In [ ]:
def bow_embed(texts):
    vocab = sorted({w.lower().strip('.,') for t in texts for w in t.split()})
    idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(texts), len(vocab)), dtype=np.float32)
    for r, t in enumerate(texts):
        for w in t.split():
            w = w.lower().strip('.,')
            if w in idx:
                M[r, idx[w]] += 1.0
    return M, idx

if HAS_ST:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    doc_emb = model.encode(corpus, normalize_embeddings=True)
    vocab_idx = None
else:
    doc_emb, vocab_idx = bow_embed(corpus)

print('matriz de embeddings:', doc_emb.shape)

## 3. Similitud coseno (numpy, correcto)

`cos(a, b) = (a·b) / (‖a‖‖b‖)`. Es la métrica estándar en retrieval denso.

In [ ]:
def cosine_sim(a, B):
    a = a / (np.linalg.norm(a) + 1e-9)
    B = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
    return B @ a

# sanity check: un vector consigo mismo da 1.0
print('cos(doc0, doc0) =', round(float(cosine_sim(doc_emb[0], doc_emb[:1])[0]), 3))

## 4. Retrieval top-k para una consulta

In [ ]:
def embed_query(q):
    if HAS_ST:
        return model.encode([q], normalize_embeddings=True)[0]
    v = np.zeros(doc_emb.shape[1], dtype=np.float32)
    for w in q.split():
        w = w.lower().strip('.,')
        if w in vocab_idx:
            v[vocab_idx[w]] += 1.0
    return v

query = '¿Qué librería uso para buscar vectores densos?'
scores = cosine_sim(embed_query(query), doc_emb)
topk = np.argsort(scores)[::-1][:3]
print('Query:', query)
for rank, i in enumerate(topk, 1):
    print(f'  {rank}. ({scores[i]:.3f}) {corpus[i]}')

## 5. Índice vectorial con FAISS (conceptual)

En producción no se recorre todo el corpus: un índice ANN (HNSW / IVF) hace la
búsqueda aproximada en millones de vectores.

In [ ]:
if False:  # requiere faiss-cpu
    import faiss
    d = doc_emb.shape[1]
    index = faiss.IndexFlatIP(d)          # inner product == coseno si esta normalizado
    index.add(doc_emb.astype('float32'))
    D, I = index.search(embed_query(query).reshape(1, -1).astype('float32'), k=3)
    print(I, D)
else:
    print('faiss.IndexFlatIP(d).add(doc_emb); index.search(q, k=3) -> (distancias, indices)')

## 6. Armar el prompt RAG (retrieve + generate)

Se inyectan los top-k como contexto y se instruye al LLM a responder **solo** con
ese contexto (mitiga alucinaciones) y a citar la fuente.

In [ ]:
context = '\n'.join(f'[{i+1}] {corpus[i]}' for i in topk)
rag_prompt = f"""Respondé usando SOLO el contexto. Si no está, decí 'No lo sé'.
Citá la fuente entre corchetes.

Contexto:
{context}

Pregunta: {query}
Respuesta:"""
print(rag_prompt)
# Este prompt se pasaria a un LLM (API o vLLM local) para generar la respuesta final.

## Ejercicios

1. **Embed + index**: embebé 100 párrafos con `all-MiniLM-L6-v2` y recuperá el
   top-5 para 3 consultas.
2. **BM25 baseline**: con `rank_bm25` compará el ranking disperso vs el denso.
3. **Hybrid (RRF)**: combiná ambos con Reciprocal Rank Fusion y verificá que
   `hybrid > dense > BM25` en consultas técnicas.
4. **Cross-encoder rerank**: reranká el top-50 con `cross-encoder/ms-marco-MiniLM-L-6-v2`.
5. **RAG con LLM**: pasá el `rag_prompt` a un LLM y verificá que cita fuentes válidas.

## Conclusiones

- RAG separa el **conocimiento** (docs, actualizable) del **modelo** (razonamiento).
- La similitud coseno sobre embeddings normalizados es la métrica base del retrieval denso.
- Un índice ANN (FAISS/HNSW) escala la búsqueda a millones de vectores.
- El prompt debe forzar "respondé solo con el contexto" para reducir alucinaciones.
- Hybrid search (denso + BM25) y cross-encoder re-ranking suben recall y precisión.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

### Ejercicio 1 — Embed + index + top-5 (API real de `sentence-transformers`)

In [ ]:
if HAS_ST:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer('all-MiniLM-L6-v2')
    emb = enc.encode(corpus, normalize_embeddings=True)      # (N, 384)
    import numpy as np
    for q in ['bases de datos vectoriales', 'recuperación dispersa']:
        qe = enc.encode([q], normalize_embeddings=True)[0]
        top = np.argsort(-(emb @ qe))[:5]
        print(q, '->', [corpus[i][:30] for i in top])
else:
    print("SentenceTransformer('all-MiniLM-L6-v2').encode(corpus) -> matriz densa;"
          " top-5 por coseno. (fallback bag-of-words ya demostrado arriba).")

### Ejercicio 2 — BM25 baseline (núcleo numpy ejecutable)

In [ ]:
# BM25: retrieval disperso basado en frecuencias (Okapi BM25).
import numpy as np, math
def bm25_scores(corpus, query, k1=1.5, b=0.75):
    docs = [d.lower().replace('.', '').replace(',', '').split() for d in corpus]
    q = query.lower().replace('¿', '').replace('?', '').split()
    N = len(docs); avgdl = sum(len(d) for d in docs) / N
    df = {}
    for d in docs:
        for w in set(d):
            df[w] = df.get(w, 0) + 1
    scores = np.zeros(N)
    for i, d in enumerate(docs):
        for w in q:
            if w not in df:
                continue
            idf = math.log(1 + (N - df[w] + 0.5) / (df[w] + 0.5))
            tf = d.count(w)
            scores[i] += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(d) / avgdl))
    return scores
query = 'librería para buscar vectores densos'
bm = bm25_scores(corpus, query)
bm_rank = list(np.argsort(-bm))
print('BM25 ranking:', [corpus[i][:28] for i in bm_rank[:3]])
assert bm_rank[0] == 0        # doc 0 (FAISS) es el mas relevante

### Ejercicio 3 — Hybrid search con Reciprocal Rank Fusion (ejecutable)

In [ ]:
# RRF fusiona rankings: score(d) = Σ 1/(k + rank_i(d)). Combina denso + BM25.
import numpy as np
def rrf(rank_lists, k=60):
    fused = {}
    for rl in rank_lists:
        for rank, idx in enumerate(rl):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(fused, key=lambda i: -fused[i])
dense = cosine_sim(embed_query(query), doc_emb)
dense_rank = list(np.argsort(-dense))
hybrid = rrf([bm_rank, dense_rank])
print('dense :', dense_rank[:3])
print('bm25  :', bm_rank[:3])
print('hybrid:', hybrid[:3])
assert int(hybrid[0]) == 0    # el doc correcto queda primero en la fusion
print('OK: hybrid mantiene arriba el documento relevante en ambas señales.')

### Ejercicio 4 — Cross-encoder re-ranking

In [ ]:
try:
    from sentence_transformers import CrossEncoder
    _CE = True
except Exception:
    _CE = False
if _CE:
    ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    pairs = [(query, corpus[i]) for i in hybrid[:5]]
    order = sorted(zip(hybrid[:5], ce.predict(pairs)), key=lambda x: -x[1])
    print('reranked:', [corpus[i][:28] for i, _ in order])
else:
    print('Cross-encoder: puntua el par (query, doc) conjuntamente (atención'
          ' cruzada) -> mucho mas preciso que el bi-encoder, pero mas caro.'
          ' Se aplica solo al top-50 del retrieval.')

### Ejercicio 5 — RAG con LLM: prompt con contexto + citas

In [ ]:
# El rag_prompt (armado arriba) se pasa a un LLM. Aqui verificamos que el
# contexto inyectado contiene el documento correcto y fuerza citar la fuente.
ctx_docs = [corpus[i] for i in topk]
assert any('FAISS' in d for d in ctx_docs)        # el doc relevante esta en contexto
assert 'SOLO el contexto' in rag_prompt            # instruccion anti-alucinacion
print('RAG prompt correcto: top-k inyectado + instruccion de citar fuentes.')
try:
    import anthropic  # noqa: F401
    print("Con API: client.messages.create(model='claude-...', "
          "messages=[{'role':'user','content': rag_prompt}])")
except Exception:
    print('(sin SDK de LLM instalado: el prompt esta listo para enviar a la API).')